# ACF & PACF: Reading the Correlogram

Reference: [/wiki/acf-pacf-interpretation](/wiki/acf-pacf-interpretation)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')


## Theoretical ACF patterns

- AR(1) φ=0.8: ρ_k = 0.8^k (geometric decay forever)
- MA(1) θ=0.7: ρ_1 = θ/(1+θ²) ≈ 0.469, ρ_k = 0 for k ≥ 2
- White noise: ρ_k ≈ 0 for all k ≥ 1


In [ ]:
rng = np.random.default_rng(42)
n = 200

# AR(1) simulation
y_ar = np.zeros(n)
for t in range(1, n):
    y_ar[t] = 0.8 * y_ar[t-1] + rng.normal()

# MA(1) simulation
eps = rng.normal(size=n+1)
y_ma = eps[1:] + 0.7 * eps[:-1]

# White noise
y_wn = rng.normal(size=n)
print('Series generated:', len(y_ar), len(y_ma), len(y_wn))


## Sample ACF from scratch

In [ ]:
def sample_acf(y, max_lag=16):
    y_c = y - y.mean()
    var = np.dot(y_c, y_c)
    return [np.dot(y_c[k:], y_c[:-k]) / var if k > 0 else 1.0
            for k in range(max_lag + 1)]

acf_ar = sample_acf(y_ar)
acf_ma = sample_acf(y_ma)
acf_wn = sample_acf(y_wn)

lags = np.arange(17)
theory_ar = 0.8 ** lags
theory_ma = np.array([1 if k==0 else 0.7/(1+0.49) if k==1 else 0 for k in lags])
sig = 1.96 / np.sqrt(n)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, acf_v, theory, title in zip(
        axes, [acf_ar, acf_ma, acf_wn], [theory_ar, theory_ma, None],
        ['AR(1) phi=0.8', 'MA(1) theta=0.7', 'White Noise']):
    ax.bar(lags, acf_v, color='steelblue', alpha=0.75, label='Sample ACF')
    if theory is not None:
        ax.plot(lags, theory, 'o--', color='orange', ms=4, label='Theory')
    ax.axhline(sig, color='yellow', ls='--', lw=1)
    ax.axhline(-sig, color='yellow', ls='--', lw=1)
    ax.set_title(title); ax.set_xlabel('Lag'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


## PACF via Yule-Walker recursion

In [ ]:
def yule_walker_pacf(y, max_lag=10):
    rho = sample_acf(y, max_lag)[1:]  # lags 1..max_lag
    pacf_vals = [rho[0]]
    for k in range(2, max_lag + 1):
        R = np.array([[rho[abs(i-j)] for j in range(k-1)] for i in range(k-1)])
        r = np.array(rho[:k-1])
        phi = np.linalg.solve(R, r) if k > 1 else np.array([])
        phi_kk = (rho[k-1] - phi @ rho[k-2::-1]) / (1 - phi @ rho[:k-1])
        pacf_vals.append(float(phi_kk))
    return pacf_vals

pacf_ar = yule_walker_pacf(y_ar)
print('PACF AR(1) lags 1-6:', [round(v,3) for v in pacf_ar[:6]])
print('Expected: large at lag 1, near-zero after that')


## Pattern guide

| ACF | PACF | Model |
|-----|------|-------|
| Exponential decay | Cuts at lag p | AR(p) |
| Cuts at lag q | Exponential decay | MA(q) |
| Both decay | Both decay | ARMA(p,q) |
| All near zero | All near zero | White noise |


## ✏️ Your turn

Compute the sample ACF at lags 1–5 for the series `y = [4, 7, 5, 8, 6, 9, 7, 10]`.


In [ ]:
y_small = np.array([4.0, 7, 5, 8, 6, 9, 7, 10])

# TODO(you): compute sample ACF at lags 1-5
# Hint: use sample_acf defined above, take [1:6]
acf_small = None  # replace

assert acf_small is not None, 'compute acf_small!'
assert len(acf_small) == 5, 'need exactly 5 values'
print('Your ACF lags 1-5:', [round(v,3) for v in acf_small])


<details>
<summary>Solution</summary>

```python
acf_small = sample_acf(y_small, max_lag=5)[1:]
```

</details>
